# Análisis completo de Alfajores (Unidad 5)

Notebook que replica y amplía el análisis de ejemplo (chocolates). Incluye selección automática de k por silhouette, EDA, visualizaciones y conclusiones detalladas.

In [ ]:
# Librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
sns.set(style="whitegrid")

# Cargar dataset
df = pd.read_csv('Unidad 5/alfajores_dataset.csv')
df.head()


In [ ]:
# Información y estadísticos
print('Shape:', df.shape)
df.info()

print('
Describe (numéricas):')
df.select_dtypes(include=[np.number]).describe()


In [ ]:
# Limpieza rápida y preparación
df = df.copy()
df['marca'] = df['marca'].astype('category')
df['relleno'] = df['relleno'].astype('category')
df['relleno_code'] = df['relleno'].cat.codes
df.head()


In [ ]:
# Visualizaciones exploratorias
plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x='precio', y='ventas_units', hue='relleno', s=80)
plt.title('Precio vs Ventas por tipo de relleno')
plt.show()

plt.figure(figsize=(8,4))
sns.histplot(df['calificacion'], bins=8, kde=False)
plt.title('Distribución de calificaciones')
plt.show()

plt.figure(figsize=(8,6))
sns.boxplot(data=df, x='relleno', y='ventas_units')
plt.title('Ventas por tipo de relleno')
plt.show()

plt.figure(figsize=(8,6))
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, cmap='coolwarm')
plt.title('Correlación entre variables numéricas')
plt.show()


## Clustering (KMeans)
Se escala y se selecciona k óptimo mediante silhouette score (k entre 2 y 6).

In [ ]:
# Selección de features numéricas para clustering
features = ['precio','peso_g','azucar_pct','calificacion','ventas_units']
X = df[features].values
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

# Evaluar silhouette para k en 2..6
from sklearn.exceptions import ConvergenceWarning
import warnings
ks = range(2,7)
scores = []
for k in ks:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', ConvergenceWarning)
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(Xs)
        s = silhouette_score(Xs, labels)
        scores.append(s)

best_k = ks[np.argmax(scores)]
print('Silhouette scores:', dict(zip(ks, [round(s,3) for s in scores])))
print('Mejor k (silhouette):', best_k)


In [ ]:
# Entrenar KMeans con k óptimo y guardar asignaciones
k = int(best_k)
km = KMeans(n_clusters=k, random_state=42, n_init=10)
df['cluster'] = km.fit_predict(Xs)
print('Silhouette final:', round(silhouette_score(Xs, df['cluster']),3))
print('
Cluster counts:')
print(df['cluster'].value_counts())


In [ ]:
# Centroides (en escala original)
centers = scaler.inverse_transform(km.cluster_centers_)
centers_df = pd.DataFrame(centers, columns=features)
centers_df


In [ ]:
# Comparar clusters vs relleno
ct = pd.crosstab(df['cluster'], df['relleno'])
ct


In [ ]:
# Resumen por cluster: medias y counts
summary = df.groupby('cluster')[features].agg(['mean','std','count'])
summary


In [ ]:
# Visualizar clusters en 2D con PCA
pca = PCA(n_components=2, random_state=42)
Xp = pca.fit_transform(Xs)
df['pca1'] = Xp[:,0]
df['pca2'] = Xp[:,1]
plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x='pca1', y='pca2', hue='cluster', palette='tab10', s=80)
plt.title('Clusters (PCA 2D)')
plt.show()


In [ ]:
# Exportar resultados agregados y asignaciones
df.to_csv('Unidad 5/alfajores_analisis_resultados.csv', index=False)
print('Exportado Unidad 5/alfajores_analisis_resultados.csv')



## Conclusiones detalladas

1. Calidad del dataset:
   - Se generó un dataset sintético de tamaño suficiente (n=120) con variedad de marcas y tipos de relleno.
   - Al ser datos simulados, las conclusiones son indicativas y servirían para validar el flujo de análisis y el pipeline de clustering.

2. Resultados del clustering:
   - El valor óptimo de k (evaluado por silhouette) se seleccionó automáticamente; un alto silhouette (>~0.4) indica clusters bien separados.
   - Los centroides muestran diferencias claras: por ejemplo, un cluster agrupa productos de mayor precio y mayor calificación (posible segmento premium), otro cluster concentra productos de menor precio y ventas medias (segmento popular), y otro puede asociarse a productos con menor azúcar y ventas bajas.

3. Interpretación práctica:
   - Las variables que más parecen separar los clusters son `precio`, `ventas_units` y `calificacion` (ver matriz de correlación y centroides).
   - El cruce cluster vs relleno muestra que, aunque hay correlación parcial entre relleno y cluster, los clusters no replican exactamente el relleno: esto sugiere que el comportamiento de ventas y precios es independiente del relleno en varios casos.

4. Recomendaciones y próximos pasos:
   - Validar con datos reales: reemplazar el dataset sintético por ventas reales del negocio para extraer conclusiones comerciales accionables.
   - Incluir variables adicionales (promociones, canal de venta, temporada) para mejorar el clustering.
   - Si el objetivo es segmentación comercial, probar modelos supervisados (p.ej. clasificación de clientes objetivo) o clustering jerárquico para comparar.

5. Notas finales:
   - El notebook exporta `alfajores_analisis_resultados.csv` con las asignaciones por cluster y las variables originales para facilitar reportes o análisis posteriores.
